<a href="https://colab.research.google.com/github/ysy3046/ai-data-bootcamp/blob/main/project_3_data_%EB%AA%A8%ED%98%95_%EA%B2%80%EC%A6%9D.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [72]:
!wget -q -O anchors.json "https://drive.google.com/uc?export=download&id=17CX9p8e6aEWp9WmksoHOtZxXzy_2ANSN"

In [73]:
!wget -q -O heritage_visitors_daily.parquet "https://drive.google.com/uc?export=download&id=1BP3zwSCFIJ5wmF5RBI-vbWBspKb9HBqI"

In [74]:
!wget -q -O sites.csv "https://drive.google.com/uc?export=download&id=10TRYm-vijCSw1If3njEf-h9i-n3KBTkP"

In [75]:
!wget -q -O weather_daily.parquet "https://drive.google.com/uc?export=download&id=1H6RmByTZH63-TGNk2Mxr6Zest-dbkB45"

In [76]:
	!wget -q -O calendar_kr_2015_2025.csv "https://drive.google.com/uc?export=download&id=1lDtkykv-Be0WEIGX4ts5Fp9MlIYULtPo"

In [77]:
!ls -lh *.parquet *.csv *.json

-rw-r--r-- 1 root root  23K Oct 12 13:02 anchors.json
-rw-r--r-- 1 root root  91K Oct 12 13:02 calendar_kr_2015_2025.csv
-rw-r--r-- 1 root root 716K Oct 12 13:02 heritage_visitors_daily.parquet
-rw-r--r-- 1 root root 8.6K Oct 12 13:02 sites.csv
-rw-r--r-- 1 root root 455K Oct 12 13:02 weather_daily.parquet


In [78]:
import pandas as pd
import json

visits = pd.read_parquet("heritage_visitors_daily.parquet")
sites = pd.read_csv("sites.csv")
weather = pd.read_parquet("weather_daily.parquet")
calendar = pd.read_csv("calendar_kr_2015_2025.csv")

with open("anchors.json", encoding="utf-8") as f:
    anchors = json.load(f)

visits.head()

,date,group,site,paid,free,domestic,foreign_en,foreign_ja,foreign_zh,foreign_etc,total
0,2015-01-01,궁,경복궁,13281,2363,10834,207,199,3907,497,15644
1,2015-01-01,궁,창덕궁,3253,847,2613,932,336,204,15,4100
2,2015-01-01,궁,창경궁,709,305,960,29,16,9,0,1014
3,2015-01-01,궁,덕수궁,1620,729,1853,98,32,344,22,2349
4,2015-01-02,궁,경복궁,10676,2028,8654,125,84,3718,123,12704


In [79]:
foreign_cols = ["foreign_en", "foreign_ja", "foreign_zh", "foreign_etc"]

assert len(visits) == anchors["rows"]
assert visits["site"].nunique() == anchors["n_sites"]
assert visits["total"].sum() == anchors["sums"]["total"]
assert visits[foreign_cols].sum().sum() == anchors["sums"]["foreign_all"]
assert (visits["total"] == 0).sum() == anchors["zero_total_site_days"]["all"]
assert len(weather) == anchors["weather"]["rows"]
assert len(calendar) == anchors["calendar"]["rows"]

print("검산 7개 모두 통과!")

검산 7개 모두 통과!


In [80]:
print("열별 결측 개수")
print(visits.isna().sum())

print("\n열별 자료형")
print(visits.dtypes)

print("\n날짜 × 개소 중복 개수")
print(visits.duplicated(["date", "site"]).sum())

열별 결측 개수
date           0
group          0
site           0
paid           0
free           0
domestic       1
foreign_en     0
foreign_ja     0
foreign_zh     0
foreign_etc    0
total          0
dtype: int64

열별 자료형
date           datetime64[ns]
group                  object
site                   object
paid                    int64
free                    int64
domestic                Int64
foreign_en              int64
foreign_ja              int64
foreign_zh              int64
foreign_etc             int64
total                   int64
dtype: object

날짜 × 개소 중복 개수
0


In [81]:
# 결측이 있는 행
display(visits[visits.isna().any(axis=1)])

# 관람객 관련 열에 음수가 있는 행
count_cols = ["paid", "free", "domestic", *foreign_cols, "total"]
display(visits[(visits[count_cols] < 0).any(axis=1)])

,date,group,site,paid,free,domestic,foreign_en,foreign_ja,foreign_zh,foreign_etc,total
25732,2021-08-15,궁,경복궁,0,0,<NA>,0,0,0,0,0


,date,group,site,paid,free,domestic,foreign_en,foreign_ja,foreign_zh,foreign_etc,total
4926,2018-05-16,궁,창경궁,-241,604,259,66,8,30,0,363
10618,2019-08-06,궁,경복궁,-1,12,11,0,0,0,0,11
26216,2021-09-06,궁,경복궁,712,513,1094,115,-1,3,14,1225


In [82]:
foreign_total = visits[foreign_cols].sum(axis=1)
known = visits["domestic"].notna()

mismatch = visits.loc[
    known & (
        visits["total"] != visits["domestic"] + foreign_total
    )
]

print("합계 불일치 행 수:", len(mismatch))
display(
    mismatch.groupby([mismatch["date"].dt.year, "site"])
    .size()
    .rename("불일치 행 수")
)

합계 불일치 행 수: 3436


date  site  
2022  김포장릉      313
      서삼릉       312
      서오릉       313
      선릉        310
      세종대왕유적    313
      양주온릉      311
      영휘원       313
      융릉        313
      의릉        313
      파주삼릉      312
      파주장릉      313
Name: 불일치 행 수, dtype: int64

In [83]:
for site, data in visits.groupby("site"):
    expected = pd.date_range(data["date"].min(), data["date"].max())
    missing = expected.difference(pd.DatetimeIndex(data["date"]))

    if len(missing):
        print(
            f"{site}: {len(missing)}일 누락 "
            f"({missing.min():%Y-%m-%d} ~ {missing.max():%Y-%m-%d})"
        )

경복궁: 53일 누락 (2020-05-30 ~ 2020-07-21)
광릉: 53일 누락 (2020-05-30 ~ 2020-07-21)
김포장릉: 53일 누락 (2020-05-30 ~ 2020-07-21)
덕수궁: 53일 누락 (2020-05-30 ~ 2020-07-21)
동구릉: 53일 누락 (2020-05-30 ~ 2020-07-21)
사릉: 53일 누락 (2020-05-30 ~ 2020-07-21)
서삼릉: 53일 누락 (2020-05-30 ~ 2020-07-21)
서오릉: 53일 누락 (2020-05-30 ~ 2020-07-21)
선릉: 53일 누락 (2020-05-30 ~ 2020-07-21)
세종대왕유적: 53일 누락 (2020-05-30 ~ 2020-07-21)
양주온릉: 53일 누락 (2020-05-30 ~ 2020-07-21)
영휘원: 53일 누락 (2020-05-30 ~ 2020-07-21)
융릉: 53일 누락 (2020-05-30 ~ 2020-07-21)
의릉: 53일 누락 (2020-05-30 ~ 2020-07-21)
정릉: 53일 누락 (2020-05-30 ~ 2020-07-21)
창경궁: 53일 누락 (2020-05-30 ~ 2020-07-21)
창덕궁: 53일 누락 (2020-05-30 ~ 2020-07-21)
태릉: 53일 누락 (2020-05-30 ~ 2020-07-21)
파주삼릉: 53일 누락 (2020-05-30 ~ 2020-07-21)
파주장릉: 53일 누락 (2020-05-30 ~ 2020-07-21)
헌릉: 53일 누락 (2020-05-30 ~ 2020-07-21)
홍유릉: 53일 누락 (2020-05-30 ~ 2020-07-21)


In [84]:
check = visits.merge(
    sites[["site", "closed_weekday"]],
    on="site",
    how="left",
    validate="many_to_one"
)

weekday_map = {0: "월", 1: "화", 2: "수", 3: "목",
               4: "금", 5: "토", 6: "일"}
check["weekday"] = check["date"].dt.dayofweek.map(weekday_map)

regular_closed = check["weekday"] == check["closed_weekday"]

print("정기 휴관 요일인데 관람객이 있는 행:",
      (regular_closed & check["total"].gt(0)).sum())

print("정기 휴관 요일이 아닌데 0명인 행:",
      (~regular_closed & check["total"].eq(0)).sum())

정기 휴관 요일인데 관람객이 있는 행: 918
정기 휴관 요일이 아닌데 0명인 행: 903


In [85]:
calendar["date"] = pd.to_datetime(calendar["date"])

check = check.merge(
    calendar[["date", "is_holiday"]],
    on="date",
    how="left",
    validate="many_to_one"
)

exceptions = check[
    check["weekday"].eq(check["closed_weekday"])
    & check["total"].gt(0)
]

print(exceptions.groupby("is_holiday").size())

is_holiday
0    277
1    641
dtype: int64


In [86]:
# 원본은 유지하고, 첫 실험용 데이터만 따로 만들기
model_data = visits.loc[
    visits["date"] >= "2023-01-01"
].copy()

model_data = model_data.sort_values(
    ["site", "date"]
).reset_index(drop=True)

print("분석 기간:",
      model_data["date"].min(),
      "~", model_data["date"].max())
print("개소 수:", model_data["site"].nunique())
print("행 수:", len(model_data))


분석 기간: 2023-01-01 00:00:00 ~ 2025-12-31 00:00:00
개소 수: 23
행 수: 25208


In [87]:
special_days = calendar.loc[
    calendar["date"].dt.year.eq(2025)
    & (
        calendar["is_holiday"].eq(1)
        | calendar["is_last_wednesday"].eq(1)
    ),
    ["date", "weekday", "holiday_name", "is_last_wednesday"]
]

display(special_days)

,date,weekday,holiday_name,is_last_wednesday
3653,2025-01-01,수,신정연휴,0
3679,2025-01-27,월,임시공휴일,0
3680,2025-01-28,화,설날 전날,0
3681,2025-01-29,수,설날,1
3682,2025-01-30,목,설날 다음날,0
3709,2025-02-26,수,NaN,1
3712,2025-03-01,토,삼일절,0
3714,2025-03-03,월,삼일절 대체 휴일,0
3737,2025-03-26,수,NaN,1
3772,2025-04-30,수,NaN,1


In [88]:
HORIZON = 7

# 예측 기간 7일이 모두 2025년 안에 들어오는 원점
origins = pd.date_range(
    start="2025-01-01",
    end="2025-12-25",
    freq="7D"
)

print("원점 수:", len(origins))
print("첫 원점:", origins.min())
print("마지막 원점:", origins.max())

원점 수: 52
첫 원점: 2025-01-01 00:00:00
마지막 원점: 2025-12-24 00:00:00


In [89]:
zero_runs = []

for site, data in visits.groupby("site"):
    data = data.sort_values("date").copy()

    # 날짜가 끊기거나, 0명 여부가 바뀌면 새 구간
    new_run = (
        data["date"].diff().ne(pd.Timedelta(days=1))
        | data["total"].eq(0).ne(data["total"].shift().eq(0))
    )

    for _, run in data.groupby(new_run.cumsum()):
        if run["total"].eq(0).all() and len(run) >= 5:
            if run["date"].max() >= pd.Timestamp("2022-01-01"):
                zero_runs.append({
                    "site": site,
                    "시작": run["date"].min(),
                    "끝": run["date"].max(),
                    "연속 일수": len(run)
                })

zero_runs_df = pd.DataFrame(zero_runs)
display(zero_runs_df)

,site,시작,끝,연속 일수
0,광릉,2024-11-27,2024-12-02,6
1,광릉,2025-07-20,2025-07-25,6
2,김포장릉,2024-11-27,2024-12-02,6
3,동구릉,2024-11-28,2024-12-02,5
4,사릉,2024-11-27,2024-12-02,6
5,서삼릉,2024-11-27,2024-12-02,6
6,서오릉,2024-11-27,2024-12-02,6
7,선릉,2024-11-27,2024-12-06,10
8,세종대왕유적,2024-11-28,2024-12-02,5
9,양주온릉,2024-11-27,2024-12-02,6


In [90]:
# 팀 기준에 맞춰 2022년부터 점검용 데이터 만들기
audit_data = visits.loc[
    visits["date"] >= "2022-01-01"
].copy()

# 행 순서가 아니라 정확히 7일 전 날짜로 연결
lag7 = audit_data[["date", "site", "total"]].copy()
lag7["date"] += pd.Timedelta(days=7)
lag7 = lag7.rename(columns={"total": "total_7days_ago"})

pairs = audit_data.merge(
    lag7, on=["date", "site"], how="left",
    validate="one_to_one"
)

valid = pairs[
    pairs["total"].gt(0)
    & pairs["total_7days_ago"].gt(0)
].copy()

valid["weekly_diff"] = (
    valid["total"] - valid["total_7days_ago"]
).abs()

display(
    valid.groupby("site")["weekly_diff"]
    .agg(유효쌍="count", 스케일="mean")
)

,유효쌍,스케일
site,,
경복궁,1249,7291.822258
광릉,1193,128.697402
김포장릉,1203,317.384871
덕수궁,1232,3385.648539
동구릉,1205,284.034025
사릉,1201,30.790175
서삼릉,1201,66.348043
서오릉,1200,467.304167
선릉,1198,617.858097


In [91]:
first_origin_scale = (
    valid.loc[valid["date"] < "2025-01-01"]
    .groupby("site")["weekly_diff"]
    .agg(유효쌍="count", 스케일="mean")
    .reindex(sites["site"])
)

display(first_origin_scale)

print(
    "분모 확인이 필요한 개소:",
    first_origin_scale.index[
        first_origin_scale["스케일"].isna()
        | first_origin_scale["스케일"].le(0)
    ].tolist()
)

,유효쌍,스케일
site,,
경복궁,943,6604.765642
창덕궁,903,1917.478405
창경궁,896,1734.959821
덕수궁,924,2992.712121
종묘,596,639.823826
동구릉,900,246.260000
광릉,898,121.688196
홍유릉,899,160.687430
사릉,896,30.389509


분모 확인이 필요한 개소: []


In [92]:
scale_checks = []

for origin in origins:
    scales = (
        valid.loc[valid["date"] < origin]
        .groupby("site")["weekly_diff"]
        .mean()
        .reindex(sites["site"])
    )

    for site, scale in scales.items():
        scale_checks.append({
            "origin": origin,
            "site": site,
            "scale": scale
        })

scale_checks_df = pd.DataFrame(scale_checks)

bad_scales = scale_checks_df[
    scale_checks_df["scale"].isna()
    | scale_checks_df["scale"].le(0)
]

print("확인한 개소 × 원점:", len(scale_checks_df))
print("분모 문제 건수:", len(bad_scales))

확인한 개소 × 원점: 1196
분모 문제 건수: 0


In [93]:
weather["date"] = pd.to_datetime(weather["date"])
calendar["date"] = pd.to_datetime(calendar["date"])

merged_check = audit_data.merge(
    weather,
    on=["date", "site"],
    how="left",
    validate="one_to_one",
    indicator="weather_match"
).merge(
    calendar,
    on="date",
    how="left",
    validate="many_to_one",
    indicator="calendar_match"
)

print("병합 전후 행 수:", len(audit_data), len(merged_check))
print("날씨 연결 실패:", merged_check["weather_match"].eq("left_only").sum())
print("달력 연결 실패:", merged_check["calendar_match"].eq("left_only").sum())

병합 전후 행 수: 33238 33238
날씨 연결 실패: 0
달력 연결 실패: 0


In [94]:
work_data = visits.loc[
    visits["date"] >= "2023-01-01"
].copy()

work_data = work_data.merge(
    sites[["site", "closed_weekday"]],
    on="site",
    how="left",
    validate="many_to_one"
).merge(
    calendar,
    on="date",
    how="left",
    validate="many_to_one"
)

# 점검용 표시: 실제 개방·휴관 확정값은 아님
work_data["is_regular_closed_day"] = (
    work_data["weekday"] == work_data["closed_weekday"]
)
work_data["is_zero_record"] = work_data["total"].eq(0)

work_data = work_data.sort_values(
    ["site", "date"]
).reset_index(drop=True)

print("행 수:", len(work_data))
print("개소 수:", work_data["site"].nunique())
print("달력 연결 실패:", work_data["weekday"].isna().sum())

행 수: 25208
개소 수: 23
달력 연결 실패: 0


In [95]:
HORIZON = 7  # 팀 초안 기준, 확정 전

origins = pd.date_range(
    start="2025-07-01",
    periods=26,
    freq="7D"
)

print("원점 수:", len(origins))
print("첫 원점:", origins.min())
print("마지막 원점:", origins.max())
print("마지막 예측일:",
      origins.max() + pd.Timedelta(days=HORIZON - 1))

원점 수: 26
첫 원점: 2025-07-01 00:00:00
마지막 원점: 2025-12-23 00:00:00
마지막 예측일: 2025-12-29 00:00:00


In [96]:
# 원본 total은 보존
work_data["target_for_model"] = work_data["total"].astype(float)
work_data.loc[
    work_data["is_zero_record"], "target_for_model"
] = float("nan")

print("학습용 값이 NaN인 행:",
      work_data["target_for_model"].isna().sum())

print("정기 휴관 요일인데 관람객이 있는 행:",
      (
          work_data["is_regular_closed_day"]
          & ~work_data["is_zero_record"]
      ).sum())

학습용 값이 NaN인 행: 3700
정기 휴관 요일인데 관람객이 있는 행: 325


In [115]:
lag7 = work_data[["date", "site", "target_for_model"]].copy()
lag7["date"] += pd.Timedelta(days=7)
lag7 = lag7.rename(columns={"target_for_model": "target_lag7"})

scale_data = work_data.merge(
    lag7, on=["date", "site"],
    how="left", validate="one_to_one"
)

scale_data["weekly_diff"] = (
    scale_data["target_for_model"] - scale_data["target_lag7"]
).abs()

scale_table = pd.concat([
    scale_data.loc[scale_data["date"] < origin]
    .groupby("site")["weekly_diff"].mean()
    .reindex(sites["site"])
    .rename("scale")
    .reset_index()
    .assign(origin=origin)
    for origin in origins
], ignore_index=True)

print("개소 × 원점:", len(scale_table))
print("분모 문제:",
      (scale_table["scale"].isna() | scale_table["scale"].le(0)).sum())

개소 × 원점: 1196
분모 문제: 0


In [116]:
baseline_rows = []

for origin in origins:
    for site, data in work_data.groupby("site"):
        history = data.loc[data["date"] < origin]
        opened = history.dropna(subset=["target_for_model"])

        if opened.empty:
            raise ValueError(f"{site}: 사용 가능한 과거 기록 없음")

        naive = opened.iloc[-1]["target_for_model"]

        recent = opened.loc[
            opened["date"] >= origin - pd.Timedelta(days=7)
        ]
        # 직전 7일에 유효 기록이 없으면 임의로 대체하지 않고 표시
        mean7 = recent["target_for_model"].mean()

        for date in pd.date_range(origin, periods=HORIZON):
            same_weekday = opened.loc[
                opened["date"].dt.dayofweek == date.dayofweek
            ]
            seasonal = (
                same_weekday.iloc[-1]["target_for_model"]
                if not same_weekday.empty else float("nan")
            )

            baseline_rows.append({
                "origin": origin,
                "site": site,
                "date": date,
                "naive": naive,
                "seasonal_naive": seasonal,
                "mean7": mean7
            })

baseline_predictions = pd.DataFrame(baseline_rows)

print("予測 행 수:", len(baseline_predictions))
print("예측값 결측:")
print(baseline_predictions[
    ["naive", "seasonal_naive", "mean7"]
].isna().sum())
display(baseline_predictions.head())

予測 행 수: 8372
예측값 결측:
naive             0
seasonal_naive    0
mean7             0
dtype: int64


,origin,site,date,naive,seasonal_naive,mean7
0,2025-01-01,경복궁,2025-01-01,17300.0,34364.0,18422.166667
1,2025-01-01,경복궁,2025-01-02,17300.0,13284.0,18422.166667
2,2025-01-01,경복궁,2025-01-03,17300.0,13348.0,18422.166667
3,2025-01-01,경복궁,2025-01-04,17300.0,15426.0,18422.166667
4,2025-01-01,경복궁,2025-01-05,17300.0,16811.0,18422.166667


In [120]:
evaluation = baseline_predictions.merge(
    work_data[["date", "site", "total"]],
    on=["date", "site"],
    how="left",
    validate="many_to_one"
).merge(
    scale_table,
    on=["origin", "site"],
    how="left",
    validate="many_to_one"
)

# 임시 평가 대상: 실제 관람객이 있는 날
evaluation = evaluation.loc[evaluation["total"].gt(0)].copy()

mase_results = []

for method in ["naive", "seasonal_naive", "mean7"]:
    evaluation[f"ae_{method}"] = (
        evaluation["total"] - evaluation[method]
    ).abs()

    by_pair = (
        evaluation.groupby(["site", "origin"])[f"ae_{method}"]
        .mean()
        .rename("MAE")
        .reset_index()
        .merge(
            scale_table,
            on=["site", "origin"],
            validate="one_to_one"
        )
    )

    by_pair["MASE"] = by_pair["MAE"] / by_pair["scale"]
    by_pair["method"] = method
    mase_results.append(by_pair)

mase_by_pair = pd.concat(mase_results, ignore_index=True)

display(
    mase_by_pair.groupby("method")
    .agg(MASE=("MASE", "mean"), 평가쌍=("MASE", "count"))
    .sort_values("MASE")
)

,MASE,평가쌍
method,,
naive,1.204833,1196
mean7,1.218304,1196
seasonal_naive,1.244695,1196


In [118]:
mase_by_origin = mase_by_pair.pivot_table(
    index="origin",
    columns="method",
    values="MASE",
    aggfunc="mean"
)

display(mase_by_origin)

method,mean7,naive,seasonal_naive
origin,,,
2025-01-01,1.418225,1.197469,1.322063
2025-01-08,0.634281,0.414627,0.733128
2025-01-15,0.492318,0.534912,0.395610
2025-01-22,1.183566,1.323161,1.163489
2025-01-29,1.313600,1.409541,2.089740
2025-02-05,1.690303,0.424149,1.588596
2025-02-12,0.673552,0.629491,0.634249
2025-02-19,0.491527,0.476646,0.398746
2025-02-26,0.755900,0.888662,0.825386


In [101]:
holiday_check = evaluation.loc[
    evaluation["origin"].isin(
        pd.to_datetime(["2025-10-07", "2025-10-14"])
    )
].copy()

# 양수: 실제보다 많이 예측 / 음수: 적게 예측
holiday_check["error_mean7"] = (
    holiday_check["mean7"] - holiday_check["total"]
)

display(
    holiday_check.groupby("origin").agg(
        평가건수=("total", "size"),
        과소예측건수=("error_mean7", lambda x: (x < 0).sum()),
        과대예측건수=("error_mean7", lambda x: (x > 0).sum()),
        평균절대오차=("ae_mean7", "mean")
    )
)

,평가건수,과소예측건수,과대예측건수,평균절대오차
origin,,,,
2025-10-07,122,92,30,6158.395589
2025-10-14,138,6,132,4098.102484


서영, 직전 7일 평균이 연휴 주에는 주로 적게, 다음 주에는 주로 많이 예측한 패턴이 확인됐어.
- 10/7 원점: 122건 중 92건 과소예측(75.4%)
- 10/14 원점: 138건 중 132건 과대예측(95.7%)
이건 개소×날짜 건수 기준이야. 명절이 원인이라고 확정한 건 아니고, 다음으로 인력 부족·과잉 비용이 얼마나 생겼는지 계산하자.

In [102]:
holiday_check["staff_gap"] = (
    holiday_check["mean7"] - holiday_check["total"]
) / 2000

holiday_check["shortage_cost"] = (
    -holiday_check["staff_gap"]
).clip(lower=0) * 600_000

holiday_check["surplus_cost"] = (
    holiday_check["staff_gap"]
).clip(lower=0) * 150_000

display(
    holiday_check.groupby("origin")[
        ["shortage_cost", "surplus_cost"]
    ].sum().round(0)
)

,shortage_cost,surplus_cost
origin,,
2025-10-07,165117493.0,15069946.0
2025-10-14,475680.0,42296441.0


In [121]:
comparison_rows = []

for method in ["naive", "seasonal_naive", "mean7"]:
    # 양수는 인력 과잉, 음수는 부족
    gap = (evaluation[method] - evaluation["total"]) / 2000

    shortage_cost = (-gap).clip(lower=0).sum() * 600_000
    surplus_cost = gap.clip(lower=0).sum() * 150_000

    comparison_rows.append({
        "방법": method,
        "MASE": mase_by_pair.loc[
            mase_by_pair["method"].eq(method), "MASE"
        ].mean(),
        "WAPE(%)": (
            evaluation[f"ae_{method}"].sum()
            / evaluation["total"].sum() * 100
        ),
        "적중률(%)": gap.ge(0).mean() * 100,
        "부족 비용(원)": shortage_cost,
        "과잉 비용(원)": surplus_cost,
        "총비용(원)": shortage_cost + surplus_cost
    })

baseline_summary = pd.DataFrame(comparison_rows)
display(baseline_summary.round(3))

,방법,MASE,WAPE(%),적중률(%),부족 비용(원),과잉 비용(원),총비용(원)
0,naive,1.205,42.840,39.986,1.476971e+09,2.015548e+08,1.678526e+09
1,seasonal_naive,1.245,49.890,50.097,1.326402e+09,3.331332e+08,1.659535e+09
2,mean7,1.218,44.417,58.189,1.191808e+09,2.938619e+08,1.485670e+09


In [122]:
CALIBRATION_WEEKS = 12
corrected_rows = []

for origin in origins:
    for site, data in work_data.groupby("site"):
        series = data.set_index("date")["target_for_model"].sort_index()
        ratios = []

        # 원점 전에 끝난 12개의 주간 예측을 재현
        for k in range(1, CALIBRATION_WEEKS + 1):
            past_origin = origin - pd.Timedelta(days=7 * k)

            past_mean = series.loc[
                (series.index >= past_origin - pd.Timedelta(days=7))
                & (series.index < past_origin)
            ].mean()

            past_actual = series.reindex(
                pd.date_range(past_origin, periods=HORIZON)
            ).dropna()

            if pd.notna(past_mean) and past_mean > 0:
                ratios.extend((past_actual / past_mean).tolist())

        if not ratios:
            raise ValueError(f"보정 자료 없음: {site}, {origin}")

        factor = pd.Series(ratios).quantile(0.8)

        current = baseline_predictions.loc[
            baseline_predictions["origin"].eq(origin)
            & baseline_predictions["site"].eq(site)
        ].copy()

        current["factor"] = factor
        current["mean7_corrected"] = current["mean7"] * factor

        corrected_rows.append(
            current[["origin", "site", "date",
                     "factor", "mean7_corrected"]]
        )

corrected_predictions = pd.concat(corrected_rows, ignore_index=True)

corrected_eval = evaluation.merge(
    corrected_predictions,
    on=["origin", "site", "date"],
    how="left",
    validate="one_to_one"
)

assert corrected_eval["mean7_corrected"].notna().all()

corrected_eval["ae_corrected"] = (
    corrected_eval["mean7_corrected"] - corrected_eval["total"]
).abs()

pair_mae = corrected_eval.groupby(
    ["site", "origin"]
)["ae_corrected"].mean()

pair_scale = scale_table.set_index(["site", "origin"])["scale"]
corrected_mase = (pair_mae / pair_scale).mean()

gap = (
    corrected_eval["mean7_corrected"] - corrected_eval["total"]
) / 2000

shortage = (-gap).clip(lower=0).sum() * 600_000
surplus = gap.clip(lower=0).sum() * 150_000

corrected_summary = pd.DataFrame([{
    "방법": "mean7 + 서비스 수준 보정",
    "MASE": corrected_mase,
    "WAPE(%)": (
        corrected_eval["ae_corrected"].sum()
        / corrected_eval["total"].sum() * 100
    ),
    "적중률(%)": gap.ge(0).mean() * 100,
    "부족 비용(원)": shortage,
    "과잉 비용(원)": surplus,
    "총비용(원)": shortage + surplus
}])

fva_first_two = pd.concat([
    baseline_summary.loc[baseline_summary["방법"].eq("mean7")],
    corrected_summary
], ignore_index=True)

fva_first_two["앞 행 대비 절감액(원)"] = (
    fva_first_two["총비용(원)"].shift()
    - fva_first_two["총비용(원)"]
)

display(fva_first_two.round(3))

,방법,MASE,WAPE(%),적중률(%),부족 비용(원),과잉 비용(원),총비용(원),앞 행 대비 절감액(원)
0,mean7,1.218,44.417,58.189,1.191808e+09,2.938619e+08,1.485670e+09,NaN
1,mean7 + 서비스 수준 보정,1.967,67.694,79.694,6.130088e+08,7.486928e+08,1.361702e+09,1.239683e+08


In [123]:
def evaluate_predictions(data, accuracy_col, staffing_col, name):
    """현재 임시 평가 규칙: 실제 total > 0인 날만 평가"""
    scored = data.loc[data["total"].gt(0)].copy()

    required = ["total", "scale", accuracy_col, staffing_col]
    if scored[required].isna().any().any():
        raise ValueError("평가 데이터에 결측이 있습니다.")
    if scored["scale"].le(0).any():
        raise ValueError("MASE 분모가 0 이하입니다.")
    if scored[staffing_col].lt(0).any():
        raise ValueError("배치용 예측값이 음수입니다.")

    scored["absolute_error"] = (
        scored[accuracy_col] - scored["total"]
    ).abs()

    # 개소 × 원점별 MAE / 학습 구간 스케일
    pair_scores = scored.groupby(["site", "origin"]).agg(
        MAE=("absolute_error", "mean"),
        scale=("scale", "first")
    )
    pair_scores["MASE"] = pair_scores["MAE"] / pair_scores["scale"]

    gap = (scored[staffing_col] - scored["total"]) / 2000
    shortage = (-gap).clip(lower=0).sum() * 600_000
    surplus = gap.clip(lower=0).sum() * 150_000

    return {
        "방법": name,
        "MASE": pair_scores["MASE"].mean(),
        "WAPE(%)": (
            scored["absolute_error"].sum()
            / scored["total"].sum() * 100
        ),
        "적중률(%)": gap.ge(0).mean() * 100,
        "부족 비용(원)": shortage,
        "과잉 비용(원)": surplus,
        "총비용(원)": shortage + surplus
    }


# 기존 결과를 같은 함수로 다시 계산
common_summary = pd.DataFrame([
    evaluate_predictions(evaluation, "naive", "naive", "naive"),
    evaluate_predictions(
        evaluation, "seasonal_naive", "seasonal_naive",
        "seasonal_naive"
    ),
    evaluate_predictions(evaluation, "mean7", "mean7", "mean7"),
    evaluate_predictions(
        corrected_eval, "mean7_corrected", "mean7_corrected",
        "mean7 + 서비스 수준 보정"
    )
])

# 기존 계산과 일치하는지 확인
import numpy as np

previous = pd.concat(
    [baseline_summary, corrected_summary], ignore_index=True
)
metrics = [
    "MASE", "WAPE(%)", "적중률(%)",
    "부족 비용(원)", "과잉 비용(원)", "총비용(원)"
]

np.testing.assert_allclose(
    common_summary[metrics].to_numpy(dtype=float),
    previous[metrics].to_numpy(dtype=float),
    rtol=1e-10,
    atol=1e-6
)

display(common_summary.round(3))
print("공통 평가 함수: 기존 결과와 일치!")

,방법,MASE,WAPE(%),적중률(%),부족 비용(원),과잉 비용(원),총비용(원)
0,naive,1.205,42.840,39.986,1.476971e+09,2.015548e+08,1.678526e+09
1,seasonal_naive,1.245,49.890,50.097,1.326402e+09,3.331332e+08,1.659535e+09
2,mean7,1.218,44.417,58.189,1.191808e+09,2.938619e+08,1.485670e+09
3,mean7 + 서비스 수준 보정,1.967,67.694,79.694,6.130088e+08,7.486928e+08,1.361702e+09


공통 평가 함수: 기존 결과와 일치!


In [106]:
import torch

print("GPU 사용 가능:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU 이름:", torch.cuda.get_device_name(0))

GPU 사용 가능: False


In [124]:
tables = []

for method in ["naive", "seasonal_naive", "mean7"]:
    table = baseline_predictions[
        ["origin", "site", "date", method]
    ].rename(columns={method: "point_pred"}).copy()

    table["model"] = method
    table["q80"] = float("nan")  # 단순 예측값은 분위수가 아님
    tables.append(table)

# 서비스 수준 보정: 점예측은 기존 평균, 배치값은 보정값
table = baseline_predictions[
    ["origin", "site", "date", "mean7"]
].rename(columns={"mean7": "point_pred"}).merge(
    corrected_predictions[
        ["origin", "site", "date", "mean7_corrected"]
    ],
    on=["origin", "site", "date"],
    validate="one_to_one"
).rename(columns={"mean7_corrected": "q80"})

table["model"] = "mean7_calibrated"
tables.append(table)

prediction_table = pd.concat(tables, ignore_index=True)[
    ["origin", "site", "date", "model", "point_pred", "q80"]
]

assert not prediction_table.duplicated(
    ["origin", "site", "date", "model"]
).any()

print("결과 행 수:", len(prediction_table))
display(prediction_table.head())

결과 행 수: 33488


,origin,site,date,model,point_pred,q80
0,2025-01-01,경복궁,2025-01-01,naive,17300.0,NaN
1,2025-01-01,경복궁,2025-01-02,naive,17300.0,NaN
2,2025-01-01,경복궁,2025-01-03,naive,17300.0,NaN
3,2025-01-01,경복궁,2025-01-04,naive,17300.0,NaN
4,2025-01-01,경복궁,2025-01-05,naive,17300.0,NaN


In [108]:
from pathlib import Path
from google.colab import files
import shutil

output_dir = Path("outputs/model_validation")
output_dir.mkdir(parents=True, exist_ok=True)

prediction_table.to_csv(
    output_dir / "predictions.csv", index=False, encoding="utf-8-sig"
)
common_summary.to_csv(
    output_dir / "baseline_summary.csv", index=False, encoding="utf-8-sig"
)
mase_by_pair.to_csv(
    output_dir / "baseline_mase_by_site_origin.csv",
    index=False, encoding="utf-8-sig"
)
scale_table.to_csv(
    output_dir / "mase_scales.csv", index=False, encoding="utf-8-sig"
)

archive = shutil.make_archive(
    "model_validation_results", "zip", root_dir=output_dir
)
files.download(archive)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [110]:
from google.colab import files

uploaded = files.upload()

Saving preprocessed_panel.csv to preprocessed_panel.csv


In [111]:
team_panel = pd.read_csv(
    "preprocessed_panel.csv",
    encoding="utf-8-sig",
    parse_dates=["date"]
)

print("행·열 수:", team_panel.shape)
print("열 목록:", team_panel.columns.tolist())
display(team_panel.head())

행·열 수: (63194, 24)
열 목록: ['date', 'site', 'group', 'weekday', 'closed_weekday', 'is_closed_weekday', 'status', 'is_open', 'y', 'total', 'paid', 'free', 'is_holiday', 'holiday_name', 'seollal_offset', 'chuseok_offset', 'is_last_wednesday', 'is_myeongjeol', 'is_free_rule', 'is_offday', 't2m', 'precip_mm', 'rain', 'in_train_period']


,date,site,group,weekday,closed_weekday,is_closed_weekday,status,is_open,y,total,...,seollal_offset,chuseok_offset,is_last_wednesday,is_myeongjeol,is_free_rule,is_offday,t2m,precip_mm,rain,in_train_period
0,2015-01-01,경복궁,궁,목,화,0,open,1,15644.0,15644.0,...,NaN,NaN,0,0,0,1,-7.77,0.03,0,0
1,2015-01-02,경복궁,궁,금,화,0,open,1,12704.0,12704.0,...,NaN,NaN,0,0,0,0,-6.94,0.31,0,0
2,2015-01-03,경복궁,궁,토,화,0,open,1,13565.0,13565.0,...,NaN,NaN,0,0,0,1,-3.83,0.00,0,0
3,2015-01-04,경복궁,궁,일,화,0,open,1,11218.0,11218.0,...,NaN,NaN,0,0,0,1,1.25,0.03,0,0
4,2015-01-05,경복궁,궁,월,화,0,open,1,7632.0,7632.0,...,NaN,NaN,0,0,0,0,1.77,1.77,0,0


In [112]:
team_recent = team_panel.loc[
    team_panel["date"] >= "2023-01-01"
].copy()

comparison = work_data[
    ["date", "site", "target_for_model"]
].merge(
    team_recent[["date", "site", "y"]],
    on=["date", "site"],
    how="outer",
    validate="one_to_one",
    indicator=True
)

same_value = np.isclose(
    comparison["target_for_model"],
    comparison["y"],
    rtol=0,
    atol=0,
    equal_nan=True
)

different = comparison.loc[
    comparison["_merge"].ne("both") | ~same_value
]

print("팀 데이터의 2023년 이후 행 수:", len(team_recent))
print("날짜·개소 또는 학습용 값 불일치:", len(different))
display(different.head())

팀 데이터의 2023년 이후 행 수: 25208
날짜·개소 또는 학습용 값 불일치: 0


,date,site,target_for_model,y,_merge


In [113]:
model_panel = team_recent.sort_values(
    ["site", "date"]
).reset_index(drop=True).copy()

print("모델용 행 수:", len(model_panel))
print("개소 수:", model_panel["site"].nunique())
print("학습용 y 결측:", model_panel["y"].isna().sum())

모델용 행 수: 25208
개소 수: 23
학습용 y 결측: 3700


In [114]:
HORIZON = 7
origins = pd.date_range(
    "2025-01-01", periods=52, freq="7D"
)

print("원점 수:", len(origins))
print("마지막 예측일:",
      origins[-1] + pd.Timedelta(days=6))

원점 수: 52
마지막 예측일: 2025-12-30 00:00:00
